# Paktech Silver transformations
Synthetic-data portfolio reconstruction. Source code is preserved from the supplied executed notebook; only explanatory markdown and metadata cleanup were added.

Full-load writes overwrite existing Silver outputs. Run in the configured Databricks workspace with ADLS-backed Unity Catalog volumes. See `docs/setup.md` for prerequisites and runtime compatibility.

Bronze: `/Volumes/paktechworkspace/default/external_volume`

Silver: `/Volumes/paktechworkspace/default/external_volume_2`

Known source-ID filters and hardcoded corrections are specific to the synthetic practice pack.

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql import Window
from pyspark.sql.functions import col
from pyspark.sql.types import *

## Branches

In [ ]:
df_branches=spark.read.format('csv').option("header", True).option("InferSchema", True).load('/Volumes/paktechworkspace/default/external_volume/branches/branches.csv')
df_branches.display()

In [ ]:
#removing duplicates

df_branches=df_branches.filter(col('_source_row_id').like('branches:000%')).orderBy(col('branch_id'))

In [ ]:
#standardizing branch_name entries and dropping two columns

df_branches=df_branches.withColumn('city', regexp_replace(col('city'), 'PEW', 'Peshawar')).drop(col('is_synthetic'), col('_extract_at'))
df_branches.display()

In [ ]:
df_branches.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/branches/branches.csv').save()

## Customers: multiple input files

In [ ]:
df_customers=spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/paktechworkspace/default/external_volume/customers/*.csv')
df_customers.display()

In [ ]:
df_customers = df_customers.filter(col('_source_row_id').like('customers:00%')).withColumn('customer_name', trim(col('customer_name')))

In [ ]:
df_customers = df_customers.withColumn('city', regexp_replace(col('city'), 'PEW', 'Peshawar')).withColumn('city', regexp_replace(col('city'), 'KHI', 'Karachi')).withColumn('city', regexp_replace(col('city'), 'LHR', 'Lahore')).withColumn('city', regexp_replace(col('city'), 'LHR', 'Lahore')).withColumn('city', regexp_replace(col('city'), 'UET', 'Lahore'))

In [ ]:
df_customers = df_customers.withColumn('email', lower(col('email'))).withColumn('email', trim(col('email'))).drop('_extract_at', 'created_at', 'updated_at', 'is_synthetic')

In [ ]:
df_customers.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/customers/customers.csv').save()

## Employees

In [ ]:
df_employees=spark.read.format('csv').option("header", True).option("InferSchema", True).load('/Volumes/paktechworkspace/default/external_volume/employees/employees.csv')
df_employees.display()

In [ ]:
df_employees = df_employees.filter(col("_source_row_id").like('employees:00%')).withColumn('display_name', trim(col('display_name'))).drop(col('active'), col('is_synthetic'), col('_extract_at'))

In [ ]:
df_employees.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/employees/employees.csv').save()

## FX rates (Silver only)

In [ ]:
df_fx_rates = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/paktechworkspace/default/external_volume/fx_rates/fx_rates.csv')
df_fx_rates.display()

In [ ]:
df_fx_rates = df_fx_rates.withColumn('rate_date', coalesce(try_to_date(col('rate_date'), 'yyyy-MM-dd'), try_to_date(col('rate_date'), 'dd-MMM-yyyy'), try_to_date(col('rate_date'), 'dd/MM/yyyy'))).drop(col("rate_basis"), col('is_synthetic'), col("_extract_at")).filter(col('_source_row_id').like('fx_rates:00%'))

In [ ]:
df_fx_rates.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/fx_rates/fx_rates.csv').save()

## Goods receipts

In [ ]:
df_goods_receipts = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/paktechworkspace/default/external_volume/goods_receipts/goods_receipts.csv')
df_goods_receipts.display()

In [ ]:
df_goods_receipts = df_goods_receipts.withColumn('receipt_date', coalesce(coalesce(try_to_date(col('receipt_date'), 'yyyy-MM-dd'), try_to_date(col('receipt_date'), 'dd/MM/yyyy'), try_to_date(col('receipt_date'), 'dd-MMM-yyyy')))).filter(col('_source_row_id').like('goods_receipts:00%')).drop(col('is_synthetic'), col('_extract_at'))

In [ ]:
df_goods_receipts.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/goods_receipts/goods_receipts.csv').save()

## Import shipments

In [ ]:
df_import_shipments = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/paktechworkspace/default/external_volume/import_shipments/import_shipments.csv')
df_import_shipments.display()

In [ ]:
df_import_shipments = df_import_shipments.filter(col('_source_row_id').like('import_shipments:00%')).drop(col('is_synthetic'), col('_extract_at'))

In [ ]:
df_import_shipments.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/import_shipments/import_shipments.csv').save()

## Installations

In [ ]:
df_installations = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/paktechworkspace/default/external_volume/installations/installations.csv')
df_installations.display()

In [ ]:
#transformations

df_installations = df_installations.filter(col('_source_row_id').like('installations:00%')).withColumn('status', lower(col('status'))).withColumn('scheduled_date', coalesce(try_to_date(col('scheduled_date'), 'yyyy-MM-dd'), try_to_date(col('scheduled_date'), 'dd/MM/yyyy'), try_to_date(col('scheduled_date'), 'dd-MMM-yyyy'))).withColumn('completed_date', coalesce(try_to_date(col('completed_date'), 'yyyy-MM-dd'), try_to_date(col('completed_date'), 'dd/MM/yyyy'), try_to_date(col('completed_date'), 'dd-MMM-yyyy'))).drop(col('updated_at'), col('is_synthetic'), col('_extract_at'))

In [ ]:
df_installations.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/installations/installations.csv').save()

## Installed assets

In [ ]:
df_installed_assets = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/paktechworkspace/default/external_volume/installed_assets/installed_assets.csv')
df_installed_assets.display()

In [ ]:
#transformations

df_installed_assets = df_installed_assets.filter(col('_source_row_id').like('installed_assets:00%')).withColumn('delivery_date', coalesce(try_to_date(col('delivery_date'), 'yyyy-MM-dd'), try_to_date(col('delivery_date'), 'dd/MM/yyyy'), try_to_date(col('delivery_date'), 'dd-MMM-yyyy'))).withColumn('warranty_end_date', coalesce(try_to_date(col('warranty_end_date'), 'yyyy-MM-dd'), try_to_date(col('warranty_end_date'), 'dd/MM/yyyy'), try_to_date(col('warranty_end_date'), 'dd-MMM-yyyy'))).drop(col('updated_at'), col('is_synthetic'), col('_extract_at'))

In [ ]:
df_installed_assets.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/installed_assets/installed_assets.csv').save()

## Invoices

In [ ]:
df_invoices = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/paktechworkspace/default/external_volume/invoices/invoices.csv')
df_invoices.display()

In [ ]:
df_invoices = df_invoices.withColumn('net_amount_pkr', when(col("net_amount_pkr").like('PKR%'), substring(col('net_amount_pkr'), 5, 15)).otherwise(col('net_amount_pkr')))

In [ ]:
df_invoices = df_invoices.withColumn('invoice_date', coalesce(try_to_date(col('invoice_date'), 'yyyy-MM-dd'), try_to_date(col('invoice_date'), 'dd/MM/yyyy'), try_to_date(col('invoice_date'), 'dd-MMM-yyyy'))).withColumn('due_date', coalesce(try_to_date(col('due_date'), 'yyyy-MM-dd'), try_to_date(col('due_date'), 'dd/MM/yyyy'), try_to_date(col('due_date'), 'dd-MMM-yyyy'))).withColumn('net_amount_pkr', regexp_replace(col('net_amount_pkr'), r'[^0-9.\-]', '')).withColumn('net_amount_pkr', col('net_amount_pkr').cast(DecimalType(20,2))).filter(col('_source_row_id').like('invoices:00%')).drop(col('tax_basis'), col('is_synthetic'), col('updated_at'), col('_extract_at'))

In [ ]:
df_invoices.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/invoices/invoices.csv').save()

## Opportunities

In [ ]:
df_opportunities = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/paktechworkspace/default/external_volume/opportunities/opportunities.csv')
df_opportunities.display()

In [ ]:
#transformations

df_opportunities = df_opportunities.withColumn('status', lower(col('status'))).withColumn('opened_date', coalesce(try_to_date(col('opened_date'), 'yyyy-MM-dd'), try_to_date(col('opened_date'), 'dd/MM/yyyy'), try_to_date(col('opened_date'), 'dd-MMM-yyyy'))).withColumn('decision_date', coalesce(try_to_date(col('decision_date'), 'yyyy-MM-dd'), try_to_date(col('decision_date'), 'dd/MM/yyyy'), try_to_date(col('decision_date'), 'dd-MMM-yyyy'))).withColumn('estimated_value_pkr', regexp_replace(col('estimated_value_pkr'), r'[^0-9.-]', '')).filter(col('_source_row_id').like('opportunities:00%')).withColumn('estimated_value_pkr', col('estimated_value_pkr').cast(DecimalType(20,2))).drop(col('updated_at'), col('is_synthetic'), col('_extract_at'))

In [ ]:
df_opportunities.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/opportunities/opportunities.csv').save()

## Payments: synthetic reference correction

In [ ]:
df_payments = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/paktechworkspace/default/external_volume/payments/payments.csv')
df_payments.display()

In [ ]:
#transformations

df_payments = df_payments.withColumn('amount_pkr', regexp_replace(col('amount_pkr'), r'[^0-9.-]', '')).withColumn('payment_date', coalesce(try_to_date(col('payment_date'), 'yyyy-MM-dd'), try_to_date(col('payment_date'), 'dd/MM/yyyy'), try_to_date(col('payment_date'), 'dd-MMM-yyyy'))).filter(col('_source_row_id').like('payments:00%')).drop(col('updated_at'), col('is_synthetic'), col('extract_at')).withColumn('amount_pkr', col('amount_pkr').cast(DecimalType(20,2)))

In [ ]:
df_payments = df_payments.withColumn(
    "amount_pkr",
    when(
        col("payment_id") == "SYN-PA-0011",
        lit("1846800.00").cast(DecimalType(20, 2))
    ).otherwise(col("amount_pkr"))
)

display(
    df_payments.filter(col("payment_id") == "SYN-PA-0011")
)

In [ ]:
df_payments.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/payments/payments.csv').save()

## Products

In [ ]:
df_products = spark.read.format('csv').option('header', True).option('InferSchema', True).load('/Volumes/paktechworkspace/default/external_volume/products/products.csv')
df_products.display()

In [ ]:
#transformations

df_products = df_products.withColumn('product_name', trim(col('product_name'))).withColumn('standard_cost_pkr', trim(col('standard_cost_pkr'))).withColumn('standard_cost_pkr', regexp_replace(col('standard_cost_pkr'), r'[^0-9.-]', '')).withColumn('standard_cost_pkr', col('standard_cost_pkr').try_cast(DecimalType(20,2))).filter(col('_source_row_id').like('products:00%')).drop(col('unit'), col('updated_at'), col('is_synthetic'), col('brand_basis'), col('_extract_at'))

In [ ]:
df_products.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/products/products.csv').save()

## Purchase order lines: duplicate-extract filtering

In [ ]:
df_purchase_order_lines = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/paktechworkspace/default/external_volume/purchase_order_lines/purchase_order_lines.csv')
df_purchase_order_lines.display()

In [ ]:
#transformation

df_purchase_order_lines = df_purchase_order_lines.withColumn('unit_landed_cost_pkr', regexp_replace(col('unit_landed_cost_pkr'), r'[^0-9.-]', '')).withColumn('unit_landed_cost_pkr', col('unit_landed_cost_pkr').cast('float')).drop(col('is_synthetic'), col('_extract_at'))

In [ ]:
df_purchase_order_lines = df_purchase_order_lines.filter(
    ~col("_source_row_id").contains(":DUP:")
)

In [ ]:
df_purchase_order_lines.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/purchase_order_lines/purchase_order_lines.csv').save()

## Purchase orders

In [ ]:
df_purchase_orders = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/paktechworkspace/default/external_volume/purchase_orders/purchase_orders.csv')
df_purchase_orders.display()

In [ ]:
#transformations

df_purchase_orders = df_purchase_orders.withColumn('order_date', coalesce(try_to_date(col('order_date'), 'yyyy-MM-dd'), try_to_date(col('order_date'), 'dd/MM/yyyy'), try_to_date(col('order_date'), 'dd-MMM-yyyy'))).withColumn('status', lower(col('status'))).filter(col('_source_row_id').like('purchase_orders:00%')).drop(col('is_synthetic'), col('updated_at'))

In [ ]:
df_purchase_orders.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/purchase_orders/purchase_orders.csv').save()

## Quotations

In [ ]:
df_quotations = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/paktechworkspace/default/external_volume/quotations/quotations.csv')
df_quotations.display()

In [ ]:
df_quotations = df_quotations.withColumn('quotation_date', coalesce(try_to_date(col('quotation_date'), 'yyyy-MM-dd'), try_to_date(col('quotation_date'), 'dd/MM/yyyy'), try_to_date(col('quotation_date'), 'dd-MMM-yyyy'))).withColumn('quoted_value_pkr', regexp_replace(col('quoted_value_pkr'), r'[^0-9.-]', '')).withColumn('quoted_value_pkr', col('quoted_value_pkr').cast('float')).filter(col('_source_row_id').like('quotations:00%')).drop(col('updated_at'), col('is_synthetic'), col('_extract_at'))

In [ ]:
df_quotations.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/quotations/quotations.csv').save()

## Sales order lines: synthetic reference corrections

In [ ]:
df_sales_order_lines = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/paktechworkspace/default/external_volume/sales_order_lines/sales_order_lines.csv')
df_sales_order_lines.display()

In [ ]:
df_sales_order_lines = df_sales_order_lines.withColumn('unit_price_pkr', regexp_replace(col('unit_price_pkr'), r'[^0-9.-]', '')).withColumn('unit_price_pkr', col('unit_price_pkr').cast('float')).withColumn('discount_rate', regexp_replace(col('discount_rate'), r'[^0-9.-]', '').alias('discount_rate_%')).filter(col('_source_row_id').like('sales_order_lines:00%')).drop('is_synthetic', col('updated_at'), col('_extract_at')).withColumnRenamed('discount_rate', 'discount_rate_%')


In [ ]:
# Corrections confirmed against the synthetic reference data.
df_sales_order_lines = (
    df_sales_order_lines
    .withColumn(
        "quantity",
        when(col("order_line_id") == "SYN-SL-0031", lit(3))
        .when(
            col("order_line_id").isin("SYN-SL-0071", "SYN-SL-0111"),
            lit(1)
        )
        .otherwise(col("quantity"))
    )
    .withColumn(
        "discount_rate_%",
        when(col("order_line_id") == "SYN-SL-0111", lit("0.05"))
        .otherwise(col("discount_rate_%"))
    )
)

In [ ]:
df_sales_order_lines = df_sales_order_lines.withColumn(
    "product_id",
    when(col("order_line_id") == "SYN-SL-0041", lit("SYN-PR-0025"))
    .when(col("order_line_id") == "SYN-SL-0081", lit("SYN-PR-0029"))
    .otherwise(col("product_id"))
)

In [ ]:
from pyspark.sql.types import DecimalType

discount = col("discount_rate_%").cast(DecimalType(10, 4))

df_sales_order_lines = df_sales_order_lines.withColumn(
    "discount_rate_%",
    when(discount > 1, discount / 100)
    .otherwise(discount)
    .cast(DecimalType(10, 4))
)

In [ ]:
df_sales_order_lines.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/sales_order_lines/sales_order_lines.csv').save()

## Sales orders

In [ ]:
df_sales_orders = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/paktechworkspace/default/external_volume/sales_orders/sales_orders.csv')
df_sales_orders.display()

In [ ]:
df_sales_orders = df_sales_orders.withColumn('order_date', coalesce(try_to_date(col('order_date'), 'yyyy-MM-dd'), try_to_date(col('order_date'), 'dd/MM/yyyy'), try_to_date(col('order_date'), 'dd-MMM-yyyy'))).withColumn('status', lower(col('status'))).filter(col('_source_row_id').like('sales_orders:00%')).drop(col('is_synthetic'), col('updated_at'), col('_extract_at'))

In [ ]:
df_sales_orders.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/sales_orders/sales_orders.csv').save()

## Service tickets

In [ ]:
df_service_tickets = spark.read.format('csv').option('header', True).option('InferSchema', True).load('/Volumes/paktechworkspace/default/external_volume/service_tickets/service_tickets.csv')
df_service_tickets.display()

In [ ]:
df_service_tickets = df_service_tickets.withColumn('closed_date', when(col('closed_date')=='N/A', None).otherwise(col('closed_date'))).withColumn('opened_date', coalesce(try_to_date(col('opened_date'), 'yyyy-MM-dd'), try_to_date(col('opened_date'), 'dd/MM/yyyy'), try_to_date(col('opened_date'), 'dd-MMM-yyyy'))).filter(col('_source_row_id').like('service_tickets:00%')).drop(col('updated_at'), col('is_synthetic'), col('_extract_at'))

In [ ]:
df_service_tickets.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/service_tickets/service_tickets.csv').save()

## Suppliers

In [ ]:
df_suppliers = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/paktechworkspace/default/external_volume/suppliers/suppliers.csv')
df_suppliers.display()

In [ ]:
df_suppliers = df_suppliers.withColumn('supplier_name', trim(col('supplier_name'))).withColumn('email', trim(lower(col('email')))).filter(col('_source_row_id').like('suppliers:00%')).drop(col('updated_at'), col('is_synthetic'), col('_extract_at'))

In [ ]:
df_suppliers.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/suppliers/suppliers.csv').save()

## Training attendance: flatten nested records

In [ ]:
df_training_attendance = spark.read.format('json').load('/Volumes/paktechworkspace/default/external_volume/training_attendance/training_attendance.jsonl')
df_training_attendance.display()

In [ ]:
from pyspark.sql.functions import explode, to_date

df_training_attendance = (
    df_training_attendance
    .withColumn("attendee", explode("attendees"))
    .select(
        "order_id",
        "training_id",
        "is_synthetic",
        "attendee.*",
        "session.*"
    )
    .withColumn("date", to_date("date", "yyyy-MM-dd"))
)

In [ ]:
df_training_attendance.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/training_attendance/training_attendance.jsonl').save()

## Training sessions

In [ ]:
df_training_sessions = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/paktechworkspace/default/external_volume/training_sessions/training_sessions.csv')
df_training_sessions.display()

In [ ]:
df_training_sessions = df_training_sessions.withColumn('session_date', coalesce(try_to_date(col('session_date'), 'yyyy-MM-dd'), try_to_date(col('session_date'), 'dd/MM/yyyy'), try_to_date(col('session_date'), 'dd-MMM-yyyy'))).filter(col('_source_row_id').like('training_sessions:00%')).drop(col('is_synthetic'), col('_extract_at'))

In [ ]:
df_training_sessions.write.format('parquet').mode('overwrite').option('path', '/Volumes/paktechworkspace/default/external_volume_2/training_sessions/training_sessions.csv').save()

In [ ]:
display(
    dbutils.fs.ls(
        "/Volumes/paktechworkspace/default/"
        "external_volume/customers/"
    )
)